# Customer Churn Prediction — Logistic Regression (From Scratch)

This notebook demonstrates a **manual implementation of Logistic Regression using NumPy**.

Scope:
- Binary classification (`Exited = 0/1`)
- No ML libraries (no scikit-learn, no torch, no TF)
- Manual gradient descent
- Manual sigmoid + loss
- Probability prediction
- Basic visualization

Pipeline:
1. Load dataset
2. Preprocess (drop, encode, scale)
3. Train/test split
4. Train model
5. Evaluate
6. Visualize probability curve

Learning purpose only.


In [19]:
import sys
import os
sys.path.append(os.path.abspath("../src"))

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from logistic_regression import LogisticRegression

df = pd.read_csv("../data/churn.csv")

print(df.dtypes)

RowNumber               int64
CustomerId              int64
Surname                   str
CreditScore             int64
Geography                 str
Gender                    str
Age                     int64
Tenure                  int64
Balance               float64
NumOfProducts           int64
HasCrCard               int64
IsActiveMember          int64
EstimatedSalary       float64
Exited                  int64
Complain                int64
Satisfaction Score      int64
Card Type                 str
Point Earned            int64
dtype: object


In [14]:
df = df.drop(columns=["RowNumber", "CustomerId", "Surname"], errors="ignore")

df["Gender"] = (df["Gender"] == "Male").astype(int)
df = pd.get_dummies(df, columns=["Geography"], drop_first=True)

# encode remaining categorical
if "Card Type" in df.columns:
    df = pd.get_dummies(df, columns=["Card Type"], drop_first=True)

df = df.astype(float)

y = df["Exited"].values
X = df.drop(columns=["Exited"]).values

# standardize
X_mean = np.mean(X, axis = 0)
X_std = np.std(X, axis = 0)
X_std[X_std == 0] = 1
X_norm = (X - X_mean) / X_std

print(X.shape)
print(y.shape)

(10000, 17)
(10000,)


In [15]:
X_train, X_test, y_train, y_test = train_test_split(
    X_norm, y,
    test_size=0.2,
    random_state=42,
    shuffle=True
)

model = LogisticRegression(learning_rate=0.001, n_epochs=1000)
model.fit(X_train, y_train)


epoch=0 loss=0.6931
epoch=100 loss=0.6748
epoch=200 loss=0.6577
epoch=300 loss=0.6416
epoch=400 loss=0.6266
epoch=500 loss=0.6124
epoch=600 loss=0.5991
epoch=700 loss=0.5866
epoch=800 loss=0.5748
epoch=900 loss=0.5636


In [16]:
y_pred = model.predict(X_test)
accuracy = np.mean(y_pred == y_test)
print("accuracy:", accuracy)
tp = np.sum((y_pred == 1) & (y_test == 1))
tn = np.sum((y_pred == 0) & (y_test == 0))
fp = np.sum((y_pred == 1) & (y_test == 0))
fn = np.sum((y_pred == 0) & (y_test == 1))

print("TP:", tp, "TN:", tn, "FP:", fp, "FN:", fn)
z = X_test @ model.weights + model.bias
proba = 1 / (1 + np.exp(-z))


accuracy: 0.948
TP: 392 TN: 1504 FP: 103 FN: 1


In [17]:
import matplotlib.pyplot as plt
import ipywidgets as widgets

proba = 1 / (1 + np.exp(-(X_test @ model.weights + model.bias)))
feature_names = df.drop(columns=["Exited"]).columns.tolist()

def plot_feature(idx):
    plt.figure()
    plt.scatter(X_test[:, idx], proba, alpha=0.3)
    plt.xlabel(feature_names[idx])
    plt.ylabel("Predicted churn probability")
    plt.title(f"Probability vs {feature_names[idx]}")
    plt.show()

widgets.interact(
    plot_feature,
    idx=widgets.IntSlider(min=0, max=X_test.shape[1]-1, step=1, value=0)
)



interactive(children=(IntSlider(value=0, description='idx', max=16), Output()), _dom_classes=('widget-interact…

<function __main__.plot_feature(idx)>